# **Prometheus Long-Term Storage: A Case Study**

## **Problem Statement**

You are a DevOps engineer at a fast-growing SaaS company. Your infrastructure monitoring system uses **Prometheus** to collect metrics from 1,000+ servers, containers, and applications.

### **Current Situation:**

- **Metrics volume:** 10,000 unique time-series
- **Scrape interval:** Every 15 seconds
- **Data generation rate:** 40,000 data points per minute (2.4 million per hour)
- **Current storage:** Prometheus local disk with 15-day retention
- **Monthly data volume:** Approximately **1-2 TB of raw metrics data**

### **The Challenge:**

Your company has new requirements:

1. **Compliance:** Must retain metrics for **12 months** for audit purposes
2. **Performance:** Dashboards and alerts must query recent data quickly (< 5 seconds)
3. **Cost:** Budget constraints require cost-effective storage solution
4. **Scalability:** System must handle 5x growth over next year
5. **Reliability:** No data loss, high availability required

### **The Question:**

**How do you architect a long-term storage solution for Prometheus that handles terabytes of metrics data efficiently?**

You're considering three approaches:

**Option A:** Use a traditional relational database (MySQL/PostgreSQL)  
**Option B:** Use a NoSQL database (MongoDB, Cassandra)  
**Option C:** Use object storage (S3) with a specialized time-series solution (Thanos/Mimir/Cortex)

---

## **Your Investigation:**

### **Key Questions You Need to Answer:**

1. **Where is the actual metrics data stored?**
   - Who handles the storage: Prometheus or Grafana?
   - What format is the data stored in?

2. **Can traditional databases handle this volume?**
   - What happens when you write millions of metrics per hour to MySQL?
   - Why would insert performance be a bottleneck?
   - How much storage overhead do databases add?

3. **How do time-series databases differ from regular databases?**
   - What makes time-series data special?
   - Why is compression so important?
   - What query patterns are common?

4. **Why do most companies use S3/object storage?**
   - What makes S3 suitable for immutable time-series blocks?
   - How does cost compare: S3 vs MySQL vs NoSQL?
   - What about query performance with remote storage?

5. **What is the recommended architecture?**
   - How does Thanos/Mimir work with Prometheus?
   - What stays local vs what goes to long-term storage?
   - How do you balance performance and cost?

---

## **Technical Deep Dive Questions:**

### **Write Performance:**

**Question:** Why can't MySQL handle 40,000 inserts per minute when it claims to support thousands of transactions per second?

<details>
<summary>Key concepts to understand:</summary>

- Transaction log overhead (WAL/redo logs)
- B-Tree index updates for every insert
- Row-level locking and contention
- Random I/O patterns vs sequential writes
- Disk IOPS limitations (even with SSDs)
</details>

---

### **Storage Efficiency:**

**Question:** You have 1 TB of raw metric data. How much storage would you actually need in MySQL vs Prometheus TSDB format?

<details>
<summary>Calculate:</summary>

- Raw data: 1 TB
- MySQL overhead: Row headers, indexes, transaction logs → **11.5 TB**
- MongoDB BSON: Document overhead, field names → **7.6 TB**
- Prometheus TSDB: Delta encoding, Gorilla compression → **87 GB**

**Why such a huge difference?**
</details>

---

### **Query Performance:**

**Question:** Query "Show CPU usage for the last 7 days for server1" - trace the execution in MySQL vs Prometheus+S3.

<details>
<summary>Performance breakdown:</summary>

**MySQL:**
1. Index scan: 2-5 seconds
2. Random row fetching: 10-30 seconds
3. String matching (LIKE): 5-10 seconds
4. Aggregation: 5-10 seconds
**Total: 22-55 seconds**

**Prometheus + S3:**
1. Block metadata lookup: 50-200ms
2. Download compressed chunks: 1-3 seconds
3. Decompress and aggregate: 200-500ms
**Total: 1.3-3.7 seconds**
</details>

---

### **Cost Analysis:**

**Question:** Compare monthly costs for storing and querying 1 TB of metrics data.

| Solution | Storage Cost | Compute Cost | Total Monthly |
|----------|-------------|--------------|---------------|
| MySQL RDS | $100 | $400-800 | **$500-900** |
| MongoDB Atlas | $200 | $300-500 | **$500-700** |
| Self-hosted Cassandra | $150 | $150-300 | **$300-450** |
| Prometheus + S3 + Thanos | $23 | $50-100 | **$73-123** |

**Why is S3 so much cheaper?**

---

## **Architecture Design Exercise:**

### **Design Task:**

Draw an architecture diagram showing:

1. **Data collection layer:** How Prometheus scrapes metrics
2. **Short-term storage:** Where recent data is stored (15 days)
3. **Long-term storage:** Where historical data goes (1+ year)
4. **Query layer:** How Grafana queries both recent and historical data
5. **Data flow:** How data moves from collection → short-term → long-term

### **Sample Architecture:**

```
┌─────────────────────────────────────────────────────────┐
│  COLLECTION LAYER                                       │
│  ┌─────────┐  ┌─────────┐  ┌─────────┐                │
│  │ Server1 │  │ Server2 │  │ ServerN │                │
│  └────┬────┘  └────┬────┘  └────┬────┘                │
│       │            │            │                       │
│       └────────────┴────────────┘                       │
│                    │ metrics (every 15s)                │
└────────────────────┼────────────────────────────────────┘
                     ▼
┌─────────────────────────────────────────────────────────┐
│  SHORT-TERM STORAGE (15 days)                           │
│  ┌──────────────────────────────────────┐              │
│  │  Prometheus                          │              │
│  │  - Local TSDB (2-hour blocks)        │              │
│  │  - In-memory head block              │              │
│  │  - Fast queries for recent data      │              │
│  └──────────────┬───────────────────────┘              │
│                 │                                        │
│                 │ remote_write (every 2 hours)          │
└─────────────────┼────────────────────────────────────────┘
                  ▼
┌─────────────────────────────────────────────────────────┐
│  LONG-TERM STORAGE (1+ year)                            │
│  ┌──────────────────────────────────────┐              │
│  │  Thanos / Mimir / Cortex             │              │
│  │  ┌────────────┐   ┌───────────────┐ │              │
│  │  │  Receiver  │───│ Store Gateway │ │              │
│  │  └────────────┘   └───────┬───────┘ │              │
│  │                            │         │              │
│  │                            ▼         │              │
│  │                    ┌───────────────┐ │              │
│  │                    │  Object Store │ │              │
│  │                    │  (S3/GCS)     │ │              │
│  │                    │               │ │              │
│  │                    │  [Block 1]    │ │              │
│  │                    │  [Block 2]    │ │              │
│  │                    │  [....]       │ │              │
│  │                    │  [Block N]    │ │              │
│  │                    └───────────────┘ │              │
│  └──────────────────────────────────────┘              │
└─────────────────┬───────────────────────────────────────┘
                  │
                  │ PromQL queries
                  │
┌─────────────────▼───────────────────────────────────────┐
│  VISUALIZATION LAYER                                    │
│  ┌──────────────────────────────────────┐              │
│  │  Grafana                             │              │
│  │  - Queries recent data from          │              │
│  │    Prometheus (< 15 days)            │              │
│  │  - Queries historical data from      │              │
│  │    Thanos (> 15 days)                │              │
│  │  - Combines results seamlessly       │              │
│  └──────────────────────────────────────┘              │
└─────────────────────────────────────────────────────────┘
```

---

## **Understanding Check Questions:**

### **Beginner Level:**

1. ✅ Does Grafana store any metrics data? (Answer: No, it only visualizes)
2. ✅ How long does Prometheus keep data locally? (Answer: 15-30 days typically)
3. ✅ What is remote_write? (Answer: Prometheus feature to send data to external storage)
4. ✅ Is Thanos free? (Answer: Yes, it's open-source)

### **Intermediate Level:**

1. ⚠️ Why is MySQL slow for time-series inserts?
2. ⚠️ What is delta encoding and why does it compress so well?
3. ⚠️ How does Thanos query old data from S3?
4. ⚠️ What is a Prometheus block?

### **Advanced Level:**

1. 🔥 Explain the complete write path from metric scrape to S3 storage
2. 🔥 Why does Prometheus use 2-hour blocks instead of streaming to S3?
3. 🔥 Calculate storage savings: 1TB raw data → Prometheus TSDB format
4. 🔥 Design a multi-region, high-availability Thanos deployment

---

## **Hands-On Exercise:**

### **Setup Lab Environment:**

```bash
# 1. Run Prometheus locally
docker run -p 9090:9090 prom/prometheus

# 2. Run Thanos sidecar with S3
docker run -v prometheus-data:/prometheus \
  -e S3_BUCKET=my-metrics \
  quay.io/thanos/thanos:latest sidecar

# 3. Run Grafana
docker run -p 3000:3000 grafana/grafana

# 4. Generate test metrics
# Write a script to generate 10,000 time-series
```

### **Tasks:**

1. ✅ Configure Prometheus with 7-day retention
2. ✅ Set up remote_write to Thanos
3. ✅ Query recent data (< 7 days) from Prometheus
4. ✅ Query old data (> 7 days) from Thanos/S3
5. ✅ Compare query performance
6. ✅ Monitor disk usage growth

---

## **Real-World Scenario:**

### **Company Growth Path:**

**Month 1:** 
- 100 servers, 10K metrics, 50 GB/month
- Solution: Prometheus local storage only
- Cost: $20/month

**Month 6:**
- 500 servers, 50K metrics, 500 GB/month
- Solution: Prometheus + Thanos + S3
- Cost: $50/month

**Month 12:**
- 1000 servers, 100K metrics, 2 TB/month
- Need: 12 months retention = 24 TB total
- Solution: Prometheus (15 days) + Thanos (11.5 months in S3)
- Cost: $120/month

**What if you used MySQL?**
- 24 TB × 11.5 bloat = 276 TB
- Cost: $5,000+/month 💸
- Performance: Queries timeout ❌

---

## **Key Takeaways (For Your Revision):**

### **Storage Layer Responsibilities:**

| Component | Role | Stores Data? | Duration |
|-----------|------|--------------|----------|
| **Prometheus** | Metrics collection + short-term storage | ✅ Yes (local TSDB) | 7-30 days |
| **Thanos/Mimir** | Long-term storage orchestration | ✅ Yes (in S3) | Months/years |
| **Grafana** | Visualization only | ❌ No | N/A |

### **Why NOT Use Traditional Databases:**

1. **Write bottleneck:** Can't handle 40K+ inserts/minute
2. **Storage bloat:** 10-15x overhead vs optimized time-series format
3. **Query slowness:** Full table scans for range queries
4. **Cost:** 10-100x more expensive than object storage

### **Why S3 + Thanos Works:**

1. **Immutable blocks:** Perfect match for Prometheus 2-hour blocks
2. **Compression:** 13:1 compression ratio (delta + Gorilla)
3. **Cost:** $0.023/GB vs $0.10-0.20/GB for databases
4. **Scalability:** Unlimited horizontal scaling
5. **Performance:** Block pruning + caching = fast queries

---

## **Interview Questions Based on This Case Study:**

1. **"How would you design a monitoring system that retains 1 year of metrics for 10,000 servers?"**

2. **"Why can't you just use PostgreSQL for storing Prometheus metrics?"**

3. **"Explain the data flow from application metrics to Grafana dashboard for 6-month-old data."**

4. **"How does Thanos achieve 13:1 compression ratio?"**

5. **"Your Prometheus disk is full after 10 days instead of 15. What would you investigate?"**

6. **"Calculate the monthly cost difference between MySQL and S3 for 5TB of metrics."**

---

## **Further Learning Path:**

### **Next Steps:**

1. ✅ Read Prometheus TSDB documentation
2. ✅ Set up a local Thanos instance
3. ✅ Study Gorilla compression algorithm
4. ✅ Practice PromQL queries
5. ✅ Design a multi-tenant metrics architecture
6. ✅ Learn about downsampling and compaction

---